# C9-dimensionality-reduction — Practice p21 — Solution

**Type:** proof · **Difficulty:** advanced · **Concepts:** pca-centered-covariance-eigenproblem-derivation

**Reasoning is required.**
Let centered $X_c\in\mathbb R^{n\times d}$ have thin SVD $X_c=U\Sigma V^{\mathsf T}$, and set $C=X_c^{\mathsf T}X_c/(n-1)$.

1. Derive $C=V(\Sigma^{\mathsf T}\Sigma/(n-1))V^{\mathsf T}$ with all shape steps stated.
2. Prove that every nonzero covariance eigenvalue is $\sigma_i^2/(n-1)$ and that the corresponding right-singular and covariance eigenspaces agree.
3. Explain separately why a simple component may flip sign and why a repeated-eigenvalue block may rotate or swap basis vectors.
4. If `Q_svd` and `Q_eigh` have orthonormal rows spanning the same repeated block, prove that comparing `Q_svd.T @ Q_svd` with `Q_eigh.T @ Q_eigh` is invariant to signs, swaps, and rotations.
5. Explain why row-by-row `np.allclose(Q_svd, Q_eigh)` is not a valid repeated-eigenspace test.

Use the sample denominator $n-1$ throughout.
The fixture below has a repeated top eigenvalue and a zero eigenvalue.
Use `ATOL = 1e-10`, `RTOL = 0.0`.


## Your proof

**Covariance/SVD expansion and shapes:**  

**Spectrum and subspace equivalence:**  

**Simple signs versus repeated rotations:**  

**Projector invariance and the failed row-wise test:**  


### Solution proof

**Covariance/SVD expansion and shapes.**  Put $r=\min(n,d)$.  In a thin SVD,
$$
U\in\mathbb R^{n\times r},\qquad
\Sigma\in\mathbb R^{r\times r},\qquad
V\in\mathbb R^{d\times r},
$$
with $U^{\mathsf T}U=I_r$, $V^{\mathsf T}V=I_r$, and
$X_c=U\Sigma V^{\mathsf T}\in\mathbb R^{n\times d}$.  Therefore
$$
X_c^{\mathsf T}X_c
=(V\Sigma^{\mathsf T}U^{\mathsf T})(U\Sigma V^{\mathsf T})
=V\Sigma^{\mathsf T}(U^{\mathsf T}U)\Sigma V^{\mathsf T}
=V\Sigma^{\mathsf T}\Sigma V^{\mathsf T},
$$
and all products have final shape $d\times d$.  Dividing by $n-1$ gives
$C=V(\Sigma^{\mathsf T}\Sigma/(n-1))V^{\mathsf T}$.

**Spectrum and subspace equivalence.**  If $v_i$ is column $i$ of $V$ and $\sigma_i>0$, then
$$
Cv_i=\frac{\sigma_i^2}{n-1}v_i.
$$
Thus every positive covariance eigenvalue is $\sigma_i^2/(n-1)$.  Conversely, the rank of both $C$ and $X_c$ is the number of positive singular values, so these account for all nonzero eigenvalues, with multiplicity.  For any fixed positive value, the span of the associated right singular vectors is therefore the same covariance eigenspace.

**Simple signs versus repeated rotations.**  A normalized eigenvector for a simple eigenvalue spans a one-dimensional eigenspace, so the only alternative normalized representative is its negative; its sign can flip.  For an eigenvalue of multiplicity $m>1$, the eigenspace is fixed but any orthonormal basis of it is valid.  Multiplying one basis by any $m\times m$ orthogonal matrix can rotate it, reflect it, change signs, or swap its vectors without changing the eigenspace.

**Projector invariance and the failed row-wise test.**  Let the orthonormal rows of $Q_{\rm svd}$ and $Q_{\rm eigh}$ span the same $m$-dimensional subspace.  There is an orthogonal $R\in\mathbb R^{m\times m}$ with
$Q_{\rm eigh}=RQ_{\rm svd}$.  Hence
$$
Q_{\rm eigh}^{\mathsf T}Q_{\rm eigh}
=Q_{\rm svd}^{\mathsf T}R^{\mathsf T}RQ_{\rm svd}
=Q_{\rm svd}^{\mathsf T}Q_{\rm svd}.
$$
This is the unique orthogonal projector onto the common subspace, so it is invariant to all signs, swaps, and rotations.  In contrast, row-by-row `np.allclose` tests a particular arbitrary basis and order.  It can fail for $Q_{\rm eigh}=RQ_{\rm svd}$ even though both computations return exactly the same eigenspace.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0
Xc_p21 = np.array([
    [np.sqrt(2.0), 0.0, 0.0],
    [-np.sqrt(2.0), 0.0, 0.0],
    [0.0, np.sqrt(2.0), 0.0],
    [0.0, -np.sqrt(2.0), 0.0],
])

_n_p21_solution = Xc_p21.shape[0]
covariance_p21 = Xc_p21.T @ Xc_p21 / (_n_p21_solution - 1)
_, _singular_values_p21, _Vt_p21 = np.linalg.svd(Xc_p21, full_matrices=False)
svd_variances_p21 = _singular_values_p21**2 / (_n_p21_solution - 1)
_eigenvalues_p21, _eigenvectors_p21 = np.linalg.eigh(covariance_p21)
_order_p21 = np.argsort(_eigenvalues_p21)[::-1]
eigh_variances_p21 = _eigenvalues_p21[_order_p21]
_Q_svd_p21 = _Vt_p21[:2]
_Q_eigh_p21 = _eigenvectors_p21[:, _order_p21[:2]].T
svd_projector_p21 = _Q_svd_p21.T @ _Q_svd_p21
eigh_projector_p21 = _Q_eigh_p21.T @ _Q_eigh_p21


In [ ]:
# Immutable contract check — do not edit.
_n_p21 = Xc_p21.shape[0]
_C_ref_p21 = Xc_p21.T @ Xc_p21 / (_n_p21 - 1)
_, _s_ref_p21, _Vt_ref_p21 = np.linalg.svd(Xc_p21, full_matrices=False)
_evals_ref_p21, _evecs_ref_p21 = np.linalg.eigh(_C_ref_p21)
_order_ref_p21 = np.argsort(_evals_ref_p21)[::-1]
_evals_ref_p21 = _evals_ref_p21[_order_ref_p21]
_Qe_ref_p21 = _evecs_ref_p21[:, _order_ref_p21[:2]].T
_Ps_ref_p21 = _Vt_ref_p21[:2].T @ _Vt_ref_p21[:2]
_Pe_ref_p21 = _Qe_ref_p21.T @ _Qe_ref_p21

for _value_p21, _shape_p21 in (
    (covariance_p21, (3, 3)),
    (svd_variances_p21, (3,)),
    (eigh_variances_p21, (3,)),
    (svd_projector_p21, (3, 3)),
    (eigh_projector_p21, (3, 3)),
):
    assert isinstance(_value_p21, np.ndarray) and _value_p21.shape == _shape_p21
    assert np.isfinite(_value_p21).all()
assert np.allclose(covariance_p21, _C_ref_p21, atol=ATOL, rtol=RTOL)
assert np.allclose(
    svd_variances_p21, _s_ref_p21**2 / (_n_p21 - 1),
    atol=ATOL, rtol=RTOL,
)
assert np.allclose(eigh_variances_p21, _evals_ref_p21, atol=ATOL, rtol=RTOL)
assert np.isclose(eigh_variances_p21[0], eigh_variances_p21[1], atol=ATOL, rtol=RTOL)
assert np.isclose(eigh_variances_p21[2], 0.0, atol=ATOL, rtol=RTOL)
assert np.allclose(svd_projector_p21, _Ps_ref_p21, atol=ATOL, rtol=RTOL)
assert np.allclose(eigh_projector_p21, _Pe_ref_p21, atol=ATOL, rtol=RTOL)
assert np.allclose(svd_projector_p21, eigh_projector_p21, atol=ATOL, rtol=RTOL)
for _P_p21 in (svd_projector_p21, eigh_projector_p21):
    assert np.allclose(_P_p21, _P_p21.T, atol=ATOL, rtol=RTOL)
    assert np.allclose(_P_p21 @ _P_p21, _P_p21, atol=ATOL, rtol=RTOL)
    assert np.isclose(np.trace(_P_p21), 2.0, atol=ATOL, rtol=RTOL)

### Answer check

The proof accounts for shapes, the squared-singular-value spectrum, sign ambiguity, repeated-block rotations, and projector invariance. The immutable fixture above checks the repeated top eigenspace and zero eigenvalue with `ATOL = 1e-10` and `RTOL = 0.0`.